# Lab 03-02: Compare Prompting Techniques

**Book chapter:** Chapter 03: Building Agents With Low-Code Tools  
**Estimated time:** 25 minutes

## Overview

Deciding whether an item can be returned requires applying several policy rules in
the right order, and Cymbal Retail's early prototypes sometimes made inconsistent
decisions. Before choosing how to instruct its agents, the team wants evidence of how
each prompting technique affects accuracy.

In this lab, you run the same six return requests, each with a known correct
decision, through four prompting techniques and measure how many decisions each one
gets right.

## Learning Objectives

By the end of this lab, you will be able to:

- Write zero-shot, few-shot, and chain-of-thought system instructions
- Request structured output that follows a schema
- Measure the accuracy of each technique against known answers
- Choose a prompting technique based on evidence rather than intuition

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents. The lab sends about 30 short requests to a Flash model and creates no resources, so there is nothing to clean up.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Technique | How it works | Use it when |
|---|---|---|
| **Zero-shot** | Instructions only, with no examples | The task is clear and common |
| **Few-shot** | Instructions plus a few example interactions | Formats or decisions are unusual or often misunderstood |
| **Chain-of-thought** | The model works through ordered steps before deciding | The task requires multi-step logic where rules interact |
| **Structured output** | The response must follow a defined schema | Code or another system must use the result |

## Setup

In [ ]:
%pip install --quiet --upgrade google-genai pydantic

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Define the Policy and the Test Cases

`POLICY` contains the five rules. `CASES` contains six requests, each paired with the
correct decision. Two cases are deliberately tricky: the boots with a split sole were
worn outdoors but are defective, so the defect rule wins; and the Gold member's gift
card is excluded even though Gold members normally have longer to return items.

In [ ]:
POLICY = """Cymbal Retail returns policy:
1. Most items: 30 days from delivery, unworn, in original packaging.
2. Footwear worn outdoors cannot be returned unless it is defective.
3. Defective items: 12 months from delivery.
4. Gold-tier members: 60 days for non-defective items.
5. Final-sale items, gift cards, and personalized products: no returns.
"""

CASES = [  # (request, correct decision)
    ("Standard member, unworn jacket, delivered 20 days ago.", "ELIGIBLE"),
    ("Standard member, unworn jacket, delivered 35 days ago.", "NOT_ELIGIBLE"),
    ("Gold member, unworn jacket, delivered 50 days ago.", "ELIGIBLE"),
    ("Boots worn on one hike, sole split open, delivered 90 days ago.",
     "ELIGIBLE"),
    ("Boots worn outdoors, no defect, delivered 10 days ago.", "NOT_ELIGIBLE"),
    ("Gold member, gift card bought 5 days ago.", "NOT_ELIGIBLE"),
]
print(f"{len(CASES)} test cases loaded.")

## Step 2: Create the Helper Functions

- `ask` sends one request with a system instruction. The temperature is 0 so that
  results are as repeatable as possible.
- `label` converts the model's text into `ELIGIBLE` or `NOT_ELIGIBLE`.
- `evaluate` runs every case through a technique, prints each decision, and returns
  the number of correct decisions.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")


def ask(system, request, temperature=0, **settings):
    config = types.GenerateContentConfig(
        system_instruction=system, temperature=temperature, **settings)
    return client.models.generate_content(
        model=MODEL, contents=request, config=config)


def label(line):
    return "NOT_ELIGIBLE" if "NOT_ELIGIBLE" in line.upper() else "ELIGIBLE"


def text_of(response):
    return (response.text or "").strip() or "(empty response)"


def evaluate(technique):
    correct = 0
    for request, expected in CASES:
        decision = technique(request)
        correct += decision == expected
        mark = "ok  " if decision == expected else "MISS"
        print(f"{mark} {decision:13} {request}")
    print(f"==> {technique.__name__}: {correct}/{len(CASES)} correct")
    return correct


scores = {}

## Step 3: Zero-Shot Prompting

The zero-shot instruction contains only the policy and the required answer format.

In [ ]:
ZERO_SHOT = POLICY + """Decide whether the return is allowed. Reply with
ELIGIBLE or NOT_ELIGIBLE on the first line, then one short reason."""


def zero_shot(request):
    return label(text_of(ask(ZERO_SHOT, request)).splitlines()[0])


scores["zero_shot"] = evaluate(zero_shot)

## Step 4: Few-Shot Prompting

The few-shot instruction adds two worked examples: an excluded item and a defective
item. Examples are most useful for decisions that the model tends to get wrong.

In [ ]:
FEW_SHOT = ZERO_SHOT + """
Examples:
Request: Standard member, personalized mug, delivered 3 days ago.
Answer: NOT_ELIGIBLE
Personalized products cannot be returned (rule 5).
Request: Standard member, faulty headlamp, delivered 4 months ago.
Answer: ELIGIBLE
Defective items can be returned for 12 months (rule 3)."""


def few_shot(request):
    return label(text_of(ask(FEW_SHOT, request)).splitlines()[0])


scores["few_shot"] = evaluate(few_shot)

## Step 5: Chain-of-Thought Prompting

The chain-of-thought instruction lists the checks in the order they must be applied.
Notice that the defect check comes **before** the worn-footwear check; that order is
what makes the split-sole boots eligible. The cell also prints the model's reasoning
for that case so that you can inspect it.

In [ ]:
CHAIN_OF_THOUGHT = POLICY + """Work through these steps in order:
1. Is the item excluded by rule 5?
2. Is the item defective? If so, apply the 12-month window.
3. Otherwise, was footwear worn outdoors? If so, it is not eligible.
4. Otherwise, apply 60 days for Gold members or 30 days for others.
Write one line per step, then a final line that contains only
ELIGIBLE or NOT_ELIGIBLE."""


def chain_of_thought(request):
    text = text_of(ask(CHAIN_OF_THOUGHT, request))
    if request == CASES[3][0]:
        print(f"Reasoning for: {request}\n{text}\n")
    return label(text.splitlines()[-1])


scores["chain_of_thought"] = evaluate(chain_of_thought)

> **Note:** In a customer-facing agent, instruct the model to reason through the steps
> internally and show the customer only the conclusion. This lab prints the reasoning
> only so that you can inspect it.

## Step 6: Structured Output

Instead of parsing free text, you can ask for a response that follows a schema. The
`Decision` class defines the fields, and the SDK returns a parsed object in
`response.parsed`. This is the most reliable option when code must act on the result.

In [ ]:
import json

from pydantic import BaseModel


class Decision(BaseModel):
    eligible: bool
    rule_number: int
    reason: str


def structured(request):
    response = ask(POLICY, request, response_mime_type="application/json",
                   response_schema=Decision)
    decision = response.parsed
    if not isinstance(decision, Decision):
        decision = Decision(**json.loads(response.text))
    return "ELIGIBLE" if decision.eligible else "NOT_ELIGIBLE"


scores["structured"] = evaluate(structured)

example = ask(POLICY, CASES[5][0], response_mime_type="application/json",
              response_schema=Decision)
print("\nExample structured response:", text_of(example))

## Step 7: Compare the Results

The next cell summarizes the scores. Because current Gemini models also reason
internally, all four techniques may score well on these simple cases. The differences
grow as rules become more numerous and ambiguous, which is why you should always test
prompts against cases with known answers.

In [ ]:
for technique, score in scores.items():
    bar = "#" * score
    print(f"{technique:17} {bar:6} {score}/{len(CASES)}")

## Step 8: Experiment

Raise the temperature to 1.0 and run the zero-shot technique twice. Higher temperatures
make responses more varied, which can make decisions less consistent between runs.

In [ ]:
def zero_shot_warm(request):
    response = ask(ZERO_SHOT, request, temperature=1.0)
    return label(text_of(response).splitlines()[0])


for run in (1, 2):
    print(f"Run {run}")
    evaluate(zero_shot_warm)
    print()

**Try it:** In Step 4, delete the faulty headlamp example from `FEW_SHOT`, run Steps 4
and 7 again, and check whether the few-shot result for the split-sole boots changes.

## Check Your Understanding

**1. Which technique would you choose if a downstream system must read the decision?**

<details><summary>Show answer</summary>

Structured output, because the response follows a schema that code can use without parsing free text.

</details>

**2. Why does the order of steps matter in the chain-of-thought instruction?**

<details><summary>Show answer</summary>

Rules interact. Checking the defect rule before the worn-footwear rule is what makes worn but defective boots eligible.

</details>

**3. Why is a temperature of 0 used for evaluation?**

<details><summary>Show answer</summary>

It makes responses as repeatable as possible, so differences in the scores come from the prompt rather than from random variation.

</details>

## Summary

In this lab, you compared zero-shot, few-shot, chain-of-thought, and structured-output
prompts on cases with known answers. You saw that examples help with decisions the model
misunderstands, that an ordered list of checks helps when rules interact, and that
schemas make results reliable for code. This lab created no resources, so there is
nothing to clean up.